# Pipeline Kaggle — Poverty SAE Niger

Entraine un ResNet-18 sur les patches Landsat 7 des 476 clusters DHS Niger 2012.

**Validation** : Group 5-fold par departement (generalisation spatiale).

**Duree** : ~2h (GPU T4/P100)
**GPU requis** : Oui (Accelerator -> GPU)

In [ ]:
import os, shutil, glob
import numpy as np
import pandas as pd

WORKING = "/kaggle/working"

# Chercher le dataset dans /kaggle/input/
KAGGLE_INPUT = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "entrainer_resnet.py" in files:
        KAGGLE_INPUT = root
        break

if KAGGLE_INPUT is None:
    raise FileNotFoundError(
        "Dataset non trouve. Cliquer sur Add Data -> Your Datasets -> poverty-sae-data"
    )
print(f"Dataset trouve : {KAGGLE_INPUT}")

In [ ]:
# Copier les fichiers dans /kaggle/working/
os.makedirs(f"{WORKING}/data/processed/patches_landsat", exist_ok=True)
os.makedirs(f"{WORKING}/data/processed", exist_ok=True)
os.makedirs(f"{WORKING}/outputs/tables", exist_ok=True)
os.makedirs(f"{WORKING}/outputs/models", exist_ok=True)
os.makedirs(f"{WORKING}/outputs/figures", exist_ok=True)

# Tous les patches (peuvent etre .tif ou .npy)
patch_src = f"{KAGGLE_INPUT}/patches_landsat"
for f in glob.glob(f"{patch_src}/*"):
    if os.path.isfile(f):
        shutil.copy(f, f"{WORKING}/data/processed/patches_landsat/")

# Script d'entrainement
shutil.copy(f"{KAGGLE_INPUT}/entrainer_resnet.py", f"{WORKING}/entrainer_resnet.py")

# Clusters GPS avec les labels de richesse
if os.path.exists(f"{KAGGLE_INPUT}/clusters_gps.csv"):
    shutil.copy(f"{KAGGLE_INPUT}/clusters_gps.csv",
                f"{WORKING}/data/processed/clusters_gps.csv")

# Afficher le nombre de fichiers
n_files = len(glob.glob(f"{WORKING}/data/processed/patches_landsat/*"))
print(f"Fichiers copies : {n_files}")

In [ ]:
# Si les patches sont en .tif, les convertir en .npy
patch_dir = f"{WORKING}/data/processed/patches_landsat"
tif_files = glob.glob(f"{patch_dir}/*.tif")
npy_files = glob.glob(f"{patch_dir}/*.npy")

if len(tif_files) > 0 and len(npy_files) == 0:
    import rasterio
    print(f"Conversion de {len(tif_files)} .tif -> .npy...")
    for tif_path in tif_files:
        basename = os.path.basename(tif_path).replace(".tif", "")
        cluster_id = int(basename.split("_")[1])

        with rasterio.open(tif_path) as src:
            patch = src.read()
        patch = np.transpose(patch, (1, 2, 0)).astype(np.float32)

        # Normalisation percentile
        p2, p98 = np.percentile(patch, [2, 98])
        patch = np.clip((patch - p2) / (p98 - p2 + 1e-8), 0, 1)
        np.save(os.path.join(patch_dir, f"{cluster_id}.npy"), patch.astype(np.float32))
    print("Conversion terminee")
else:
    print("Patches deja en .npy, conversion ignoree")

In [ ]:
# Verifier que tout est en ordre avant l'entrainement
npy_files = glob.glob(f"{patch_dir}/*.npy")
has_labels = os.path.exists(f"{WORKING}/data/processed/clusters_gps.csv")
print(f"Patches .npy : {len(npy_files)}")
print(f"Labels : {'OK' if has_labels else 'MANQUANT'}")
assert len(npy_files) > 0, "Aucun patch trouve"
assert has_labels, "clusters_gps.csv manquant"
print("Pret pour l'entrainement")

In [ ]:
# Installer les dependances
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118 -q
!pip install scikit-learn pandas matplotlib rasterio -q
print("Dependances OK")

## Entrainement ResNet-18

Validation croisee spatiale (Group 5-fold par departement).
Courbes d'apprentissage + predictions pour le modele FH.

In [ ]:
os.chdir(WORKING)
!python entrainer_resnet.py

## Resultats

Tout est dans `/kaggle/working/`. Pour telecharger :

In [ ]:
!zip -r results.zip outputs/ data/processed/cnn_predictions_cluster.csv
print("Fichiers zippes dans results.zip")
print("Telecharger depuis le panneau Output (a droite)")